In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from glob import glob
from PIL import Image
import pandas as pd
from skimage.morphology import skeletonize
from skimage.measure import label, regionprops
from tensorflow.keras.utils import load_img, img_to_array
from skimage.morphology import remove_small_objects, remove_small_holes, binary_closing, disk

In [ ]:
import os

os.environ["KAGGLE_API_TOKEN"] = "KGAT_d2a760e9c0b21e8e5d2147e657f460d7"

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('umud-challenge-muscle-architecture-in-ultrasound-data')

print("Path to competition files:", path)

100%|██████████| 2.56G/2.56G [00:28<00:00, 96.6MB/s]

Extracting files...


Path to competition files: /root/.cache/kagglehub/competitions/umud-challenge-muscle-architecture-in-ultrasound-data


In [ ]:
base = "/root/.cache/kagglehub/competitions/umud-challenge-muscle-architecture-in-ultrasound-data"

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving apo_model_dice.keras to apo_model_dice.keras
Saving fascicle_model.keras to fascicle_model.keras


In [ ]:
# Dice coefficient measures how much the predicted mask overlaps with the true mask
# Higher Dice is better, with 1 meaning perfect overlap
def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    # Flatten the masks so we compare all pixels together
    y_true = tf.reshape(y_true, [-1])
    y_pred = tf.reshape(y_pred, [-1])

    # Count the overlapping pixels between the true and predicted masks
    intersection = tf.reduce_sum(y_true * y_pred)

    # Compute Dice score
    return (2.0 * intersection + smooth) / (
        tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) + smooth
    )

# Dice loss is the opposite of Dice score
# Lower Dice loss means better segmentation
def dice_loss(y_true, y_pred):
    return 1.0 - dice_coef(y_true, y_pred)

# Combine binary crossentropy and Dice loss
# BCE helps with pixel-wise classification
# Dice helps with mask overlap
def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    dice = dice_loss(y_true, y_pred)

    return bce + dice

In [ ]:
apo_model_dice = tf.keras.models.load_model(
    "/content/apo_model_dice.keras",
    custom_objects={
        "bce_dice_loss": bce_dice_loss,
        "dice_coef": dice_coef
    }
)

fas_model_dice = tf.keras.models.load_model(
    "/content/fascicle_model.keras",
    custom_objects={
        "bce_dice_loss": bce_dice_loss,
        "dice_coef": dice_coef
    }
)

In [ ]:
test_images = sorted(
    glob("/root/.cache/kagglehub/competitions/umud-challenge-muscle-architecture-in-ultrasound-data/test_images_v2/test_set_v2/*")
)

In [ ]:
competition_dir = "/root/.cache/kagglehub/competitions/umud-challenge-muscle-architecture-in-ultrasound-data"

In [1]:
from tensorflow.keras.utils import load_img, img_to_array

IMG_SIZE = 512

def load_tiff_python(img_path, mask_path):
    img_path = img_path.numpy().decode("utf-8")
    mask_path = mask_path.numpy().decode("utf-8")

    img = load_img(
        img_path,
        color_mode="grayscale",
        target_size=(IMG_SIZE, IMG_SIZE)
    )
    img = img_to_array(img).astype("float32") / 255.0

    mask = load_img(
        mask_path,
        color_mode="grayscale",
        target_size=(IMG_SIZE, IMG_SIZE)
    )
    mask = img_to_array(mask).astype("float32") / 255.0

    mask = (mask > 0.5).astype("float32")

    return img, mask


def load_image_mask(img_path, mask_path):
    img, mask = tf.py_function(
        func=load_tiff_python,
        inp=[img_path, mask_path],
        Tout=[tf.float32, tf.float32]
    )

    img.set_shape((IMG_SIZE, IMG_SIZE, 1))
    mask.set_shape((IMG_SIZE, IMG_SIZE, 1))

    return img, mask

In [2]:
from sklearn.model_selection import train_test_split
import glob

# Training image paths
apo_imgs = sorted(glob.glob(base + "/apo_imgs_v1/**/*.tif", recursive=True))
apo_masks = sorted(glob.glob(base + "/apo_masks_v1/**/*.tif", recursive=True))

fasc_imgs = sorted(glob.glob(base + "/fasc_imgs_v1/**/*.tif", recursive=True))
fasc_masks = sorted(glob.glob(base + "/fasc_masks_v1/**/*.tif", recursive=True))

# Same split used during training
_, apo_val_imgs, _, apo_val_masks = train_test_split(
    apo_imgs,
    apo_masks,
    test_size=0.2,
    random_state=42
)

_, fas_val_imgs, _, fas_val_masks = train_test_split(
    fasc_imgs,
    fasc_masks,
    test_size=0.2,
    random_state=42
)

BATCH_SIZE = 8

val_ds = (
    tf.data.Dataset.from_tensor_slices((apo_val_imgs, apo_val_masks))
    .map(load_image_mask)
    .batch(BATCH_SIZE)
)

fas_val_ds = (
    tf.data.Dataset.from_tensor_slices((fas_val_imgs, fas_val_masks))
    .map(load_image_mask)
    .batch(BATCH_SIZE)
)

NameError: name 'base' is not defined

In [ ]:
# -----------------------------
# Segmentation metrics
# -----------------------------

def compute_segmentation_metrics(model, dataset, threshold=0.5):
    y_true_all = []
    y_pred_all = []

    for imgs, masks in dataset:
        preds = model.predict(imgs, verbose=0)

        y_true = masks.numpy().astype(np.uint8)
        y_pred = (preds > threshold).astype(np.uint8)

        y_true_all.append(y_true.reshape(-1))
        y_pred_all.append(y_pred.reshape(-1))

    y_true_all = np.concatenate(y_true_all)
    y_pred_all = np.concatenate(y_pred_all)

    tp = np.sum((y_true_all == 1) & (y_pred_all == 1))
    fp = np.sum((y_true_all == 0) & (y_pred_all == 1))
    fn = np.sum((y_true_all == 1) & (y_pred_all == 0))
    tn = np.sum((y_true_all == 0) & (y_pred_all == 0))

    accuracy = (tp + tn) / (tp + fp + fn + tn + 1e-7)
    precision = tp / (tp + fp + 1e-7)
    recall = tp / (tp + fn + 1e-7)
    dice = (2 * tp) / (2 * tp + fp + fn + 1e-7)
    iou = tp / (tp + fp + fn + 1e-7)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "dice": dice,
        "iou": iou
    }

In [ ]:
# -----------------------------
# Compute metrics for both models
# -----------------------------

apo_metrics = compute_segmentation_metrics(
    apo_model_dice,
    val_ds
)

fas_metrics = compute_segmentation_metrics(
    fas_model_dice,
    fas_val_ds
)

performance_df = pd.DataFrame({
    "Aponeurosis Model": apo_metrics,
    "Fascicle Model": fas_metrics
})

performance_df

,Aponeurosis Model,Fascicle Model
accuracy,0.983817,0.994235
precision,0.747841,0.184705
recall,0.748079,0.282082
dice,0.747960,0.223236
iou,0.597393,0.125642
